# MOD 11.1 – Projet 4 : Gestion des conférences
## Alimentation de la base `gestion_conferences`

Ce notebook remplit la base avec des données **réalistes et en volume suffisant** pour que les 9 requêtes du sujet donnent des résultats pertinents.

**Principes :**
- **Reproductible** : graine aléatoire fixe (`SEED = 42`), on obtient toujours les mêmes données.
- **Ré-exécutable** : la cellule 3 vide les tables avant de les remplir.
- **Cohérent** : toutes les données respectent les contraintes (CHECK, FK) **et les triggers** de `02_triggers.sql`. Si une donnée était incohérente, PostgreSQL refuserait l'insertion.
- **Cas de test volontaires** pour chaque requête : série SDH, professeur « X », thèmes spatiaux sans le mot « spatial », conférences de l'année courante, utilisateurs qui sont aussi responsables.

**Prérequis :** la base est créée (`./init_db.sh`) et les paquets sont installés :
```bash
pip install psycopg2-binary faker werkzeug jupyter
```

## 1. Imports et paramètres

In [34]:
import random
from datetime import date, timedelta

import psycopg2
from psycopg2.extras import execute_values
from faker import Faker
from werkzeug.security import generate_password_hash

SEED = 42
random.seed(SEED)
Faker.seed(SEED)
fake = Faker(['fr_FR', 'en_US', 'de_DE', 'it_IT', 'es_ES'])   # noms de plusieurs nationalités

# Connexion : adapter si besoin (utilisateur, mot de passe, hôte)
DB_PARAMS = dict(dbname='gestion_conferences')

ANNEE_COURANTE = date.today().year
ANNEES = range(ANNEE_COURANTE - 6, ANNEE_COURANTE + 2)   # 6 ans d'historique + l'an prochain

# Volumes
NB_CONFERENCES   = 160   # conférences "principales" (hors workshops)
NB_WORKSHOPS     = 45
NB_PERSONNES     = 320   # chercheurs (responsables, membres de comités)
NB_UTIL_EXTERNES = 45    # utilisateurs qui ne sont que des utilisateurs
NB_UTIL_CHERCHEURS = 25  # utilisateurs qui sont AUSSI des chercheurs (requête 8)
print('Année courante :', ANNEE_COURANTE)

Année courante : 2026


## 2. Connexion et fonctions utilitaires

In [35]:
conn = psycopg2.connect(**DB_PARAMS)
conn.autocommit = False
cur = conn.cursor()
cur.execute("SET search_path TO conferences, public")

def inserer(table, colonnes, lignes):
    """Insère une liste de tuples en une seule requête (rapide)."""
    if not lignes:
        return
    sql = f"INSERT INTO {table} ({', '.join(colonnes)}) VALUES %s"
    execute_values(cur, sql, lignes, page_size=1000)
    print(f'  {table:<22} : {len(lignes):>5} lignes')

print('Connecté à', conn.get_dsn_parameters()['dbname'])

Connecté à gestion_conferences


## 3. Vider les tables
`TRUNCATE ... CASCADE` vide toutes les tables d'un coup, `RESTART IDENTITY` remet les compteurs `SERIAL` à 1.

In [36]:
cur.execute("""
    TRUNCATE pays, ville, organisme, serie, editeur, type_responsabilite,
             categorie_soumission, theme, conference, workshop, personne,
             utilisateur, session, responsable_de, membre_pc, evalue,
             session_theme, utilisateur_theme, specification
    RESTART IDENTITY CASCADE
""")
print('Tables vidées')

Tables vidées


## 4. Référentiels : pays, villes, organismes, séries, éditeurs, types, catégories

In [37]:
GEO = {
    'France':        ['Lyon', 'Paris', 'Toulouse', 'Grenoble', 'Nantes'],
    'Allemagne':     ['Berlin', 'Munich', 'Hambourg'],
    'Italie':        ['Rome', 'Milan', 'Turin'],
    'Espagne':       ['Madrid', 'Barcelone', 'Valence'],
    'Royaume-Uni':   ['Londres', 'Édimbourg', 'Manchester'],
    'Pays-Bas':      ['Amsterdam', 'Delft'],
    'Belgique':      ['Bruxelles', 'Louvain'],
    'Suisse':        ['Zurich', 'Genève', 'Lausanne'],
    'Canada':        ['Montréal', 'Toronto', 'Vancouver'],
    'États-Unis':    ['New York', 'San Francisco', 'Boston', 'Seattle'],
    'Japon':         ['Tokyo', 'Kyoto'],
    'Chine':         ['Pékin', 'Shanghai'],
    'Brésil':        ['São Paulo', 'Rio de Janeiro'],
    'Tunisie':       ['Tunis', 'Sousse'],
    'Australie':     ['Sydney', 'Melbourne'],
}
pays, villes = [], []
for i, (nom_pays, liste) in enumerate(GEO.items(), start=1):
    pays.append((i, nom_pays))
    for nom_ville in liste:
        villes.append((len(villes) + 1, nom_ville, i))

UNIVERSITES = [
    ('École Centrale de Lyon', 'Lyon'), ('Université Claude Bernard Lyon 1', 'Lyon'),
    ('Sorbonne Université', 'Paris'), ('Université Paris-Saclay', 'Paris'),
    ('Université de Toulouse', 'Toulouse'), ('Université Grenoble Alpes', 'Grenoble'),
    ('TU Berlin', 'Berlin'), ('TU München', 'Munich'), ('Sapienza Università di Roma', 'Rome'),
    ('Politecnico di Milano', 'Milan'), ('Universidad Politécnica de Madrid', 'Madrid'),
    ('Universitat de Barcelona', 'Barcelone'), ('University of Edinburgh', 'Édimbourg'),
    ('Imperial College London', 'Londres'), ('TU Delft', 'Delft'), ('KU Leuven', 'Louvain'),
    ('ETH Zürich', 'Zurich'), ('EPFL', 'Lausanne'), ('Université de Montréal', 'Montréal'),
    ('University of Toronto', 'Toronto'), ('MIT', 'Boston'), ('Stanford University', 'San Francisco'),
    ('University of Tokyo', 'Tokyo'), ('Tsinghua University', 'Pékin'),
    ('Universidade de São Paulo', 'São Paulo'), ('Université de Tunis El Manar', 'Tunis'),
    ('University of Sydney', 'Sydney'),
]
id_ville = {v[1]: v[0] for v in villes}
organismes = []
for i, (nom, ville) in enumerate(UNIVERSITES, start=1):
    domaine = ''.join(c for c in nom.lower() if c.isalnum())[:12]
    organismes.append((i, nom, f'{fake.street_address()}, {ville}', f'conferences@{domaine}.edu'))
ville_organisme = {o[0]: id_ville[UNIVERSITES[o[0] - 1][1]] for o in organismes}

SERIES = [
    ('SDH',   'International Symposium on Spatial Data Handling'),
    ('CAISE', 'Conference on Advanced Information Systems Engineering'),
    ('VLDB',  'International Conference on Very Large Data Bases'),
    ('ICDE',  'International Conference on Data Engineering'),
    ('ER',    'International Conference on Conceptual Modeling'),
    ('DEXA',  'Database and Expert Systems Applications'),
    ('EDBT',  'International Conference on Extending Database Technology'),
    ('TIME',  'Symposium on Temporal Representation and Reasoning'),
    ('BDA',   'Gestion de Données – Principes, Technologies et Applications'),
]
series = [(i, s, n) for i, (s, n) in enumerate(SERIES, start=1)]

EDITEURS = ['Springer', 'IEEE', 'ACM', 'Elsevier', 'CEUR-WS', 'IOS Press', 'LIPIcs', 'OpenProceedings']
editeurs = [(i, e) for i, e in enumerate(EDITEURS, start=1)]

TYPES_RESP = ['General chair', 'Program Committee chair', 'Steering Committee',
              'Poster chair', 'Industrial session chair', 'Workshop chair', 'Publication chair']
types_resp = [(i, t) for i, t in enumerate(TYPES_RESP, start=1)]

CATEGORIES = ['Regular paper', 'Short paper', 'Panel', 'Tutorial', 'Industrial development',
              'System description', 'Poster', 'Workshop proposal', 'Demo']
SANS_PAGES = {'Panel', 'Tutorial', 'Workshop proposal'}       # pas de nombre de pages (sujet)
categories = [(i, c) for i, c in enumerate(CATEGORIES, start=1)]

inserer('pays', ['id_pays', 'nom_pays'], pays)
inserer('ville', ['id_ville', 'nom_ville', 'id_pays'], villes)
inserer('organisme', ['id_organisme', 'nom', 'adresse', 'email_contact'], organismes)
inserer('serie', ['id_serie', 'sigle', 'nom_serie'], series)
inserer('editeur', ['id_editeur', 'nom_editeur'], editeurs)
inserer('type_responsabilite', ['id_type', 'libelle'], types_resp)
inserer('categorie_soumission', ['id_categorie', 'libelle'], categories)

  pays                   :    15 lignes
  ville                  :    41 lignes
  organisme              :    27 lignes
  serie                  :     9 lignes
  editeur                :     8 lignes
  type_responsabilite    :     7 lignes
  categorie_soumission   :     9 lignes


## 5. Thèmes de recherche (hiérarchie)
Pour la **requête 3**, la branche « Spatial databases » contient volontairement des sous-thèmes **qui ne contiennent pas le mot "spatial"** (GIS, R-tree, Trajectory data…). Une simple recherche `LIKE '%spatial%'` les raterait ; la recherche récursive dans la hiérarchie les trouve.

In [38]:
HIERARCHIE = {
    'Spatial databases': {
        'GIS': ['Geographic information retrieval', 'Cartographic generalization'],
        'Spatial indexing': ['R-tree', 'Quadtree'],
        'Spatio-temporal data': ['Trajectory data', 'Moving objects'],
        'Location-based services': [],
        'Remote sensing data': [],
    },
    'Temporal databases': {
        'Temporal data models': [], 'Temporal database design': [],
        'Temporal query languages': [], 'Indexing of temporal data': [],
    },
    'Data management': {
        'Query optimization': [], 'Transaction processing': [],
        'NoSQL databases': ['Graph databases', 'Document stores'],
        'Data integration': [], 'Data quality': [],
    },
    'Information systems engineering': {
        'Conceptual modeling': ['Entity-relationship model'],
        'Business process management': [], 'Requirements engineering': [],
    },
    'Artificial intelligence': {
        'Machine learning': ['Deep learning', 'Reinforcement learning'],
        'Natural language processing': [], 'Knowledge graphs': [],
    },
    'Big data': {
        'Distributed computing': [], 'Stream processing': [], 'Data lakes': [],
    },
    'Security and privacy': {
        'Access control': [], 'Data anonymization': [],
    },
}
themes = []          # (id_theme, libelle, id_parent)
def ajouter_theme(libelle, parent):
    themes.append((len(themes) + 1, libelle, parent))
    return len(themes)

for racine, enfants in HIERARCHIE.items():
    id_r = ajouter_theme(racine, None)
    for enfant, petits_enfants in enfants.items():
        id_e = ajouter_theme(enfant, id_r)
        for pe in petits_enfants:
            ajouter_theme(pe, id_e)

# Les parents doivent exister avant les enfants : l'ordre de la liste le garantit
inserer('theme', ['id_theme', 'libelle', 'id_theme_parent'], themes)

# Pour choisir des thèmes cohérents : thèmes de chaque branche (racine + descendants)
id_racine = {}
for t_id, lib, parent in themes:
    id_racine[t_id] = t_id if parent is None else id_racine[parent]
branches = {}
for t_id, r in id_racine.items():
    branches.setdefault(r, []).append(t_id)
ID_SPATIAL = next(t[0] for t in themes if t[1] == 'Spatial databases')
ID_TEMPORAL = next(t[0] for t in themes if t[1] == 'Temporal databases')

  theme                  :    43 lignes


## 6. Personnes et utilisateurs
- 320 chercheurs, dont le **professeur « X »** de la requête 9 : *Jean Dupont*.
- 25 chercheurs ont **aussi** un compte utilisateur (requête 8).
- 45 utilisateurs « externes » (doctorants, ingénieurs…) qui ne sont responsables de rien.
- Deux comptes de démonstration pour l'interface : `admin / admin123` et `alice / alice123`.
- Les mots de passe sont **hashés** (jamais stockés en clair).

In [39]:
import unicodedata

def sans_accents(texte):
    """'Rómulo Clément' -> 'romulo clement' (pour les emails et les logins)."""
    texte = unicodedata.normalize('NFKD', texte).encode('ascii', 'ignore').decode()
    return ''.join(c for c in texte.lower() if c.isalnum() or c == '.')

def email_unique(prenom, nom, deja):
    base = sans_accents(f'{prenom}.{nom}') or 'user'
    email, n = f'{base}@{random.choice(["univ-lyon.fr", "uni.edu", "research.org", "ec-lyon.fr", "lab.eu"])}', 1
    while email in deja:
        n += 1
        email = f'{base}{n}@research.org'
    deja.add(email)
    return email

emails = set()
personnes = []   # (id, nom, prenom, adresse_pro, email)

# Le professeur "X" de la requête 9
personnes.append((1, 'Dupont', 'Jean', 'École Centrale de Lyon, 36 av. Guy de Collongue, Ecully',
                  email_unique('jean', 'dupont', emails)))
# La personne de démonstration (utilisatrice ET chercheuse)
personnes.append((2, 'Martin', 'Alice', 'Université Claude Bernard Lyon 1, Villeurbanne',
                  email_unique('alice', 'martin', emails)))

for i in range(3, NB_PERSONNES + 1):
    prenom, nom = fake.first_name(), fake.last_name()
    univ = random.choice(UNIVERSITES)[0]
    personnes.append((i, nom, prenom, f'{univ}, {fake.street_address()}', email_unique(prenom, nom, emails)))
CHERCHEURS = [p[0] for p in personnes]

# Utilisateurs externes : de nouvelles personnes, sans adresse professionnelle
id_admin = NB_PERSONNES + 1
personnes.append((id_admin, 'Admin', 'Système', None, 'admin@gestion-conferences.fr'))
emails.add('admin@gestion-conferences.fr')
externes = []
for i in range(id_admin + 1, id_admin + 1 + NB_UTIL_EXTERNES):
    prenom, nom = fake.first_name(), fake.last_name()
    personnes.append((i, nom, prenom, None, email_unique(prenom, nom, emails)))
    externes.append(i)

inserer('personne', ['id_personne', 'nom', 'prenom', 'adresse_pro', 'email'], personnes)

# Comptes utilisateurs
hash_defaut = generate_password_hash('user123')     # même mot de passe pour les comptes générés
utilisateurs = [
    (id_admin, 'admin', generate_password_hash('admin123'), 'admin'),
    (2,        'alice', generate_password_hash('alice123'), 'utilisateur'),
]
chercheurs_util = random.sample(CHERCHEURS[2:], NB_UTIL_CHERCHEURS - 1)   # Alice est déjà comptée
for pid in chercheurs_util + externes:
    p = personnes[pid - 1]
    login = f'{sans_accents(p[2])[:1]}{sans_accents(p[1]).replace(".", "")}{pid}'
    utilisateurs.append((pid, login, hash_defaut, 'utilisateur'))
inserer('utilisateur', ['id_personne', 'login', 'mot_de_passe', 'role'], utilisateurs)

  personne               :   366 lignes
  utilisateur            :    71 lignes


## 7. Conférences principales
- Réparties entre l'année courante − 6 et l'année courante + 1 ; **au moins 15 conférences ont lieu l'année courante** (requête 1).
- Environ 70 % appartiennent à une série ; la série **SDH** a une édition par an (requête 5).
- Chaque conférence se tient dans la ville de son organisme ou dans une autre ville.

In [40]:
SUJETS = {
    ID_SPATIAL:  ['Spatial Information', 'Geographic Data Science', 'Location Intelligence', 'GIScience'],
    ID_TEMPORAL: ['Temporal Data', 'Time Series Management', 'Temporal Reasoning'],
}
SUJETS_GENERAUX = ['Data Engineering', 'Information Systems', 'Database Systems', 'Data Science',
                   'Knowledge Management', 'Big Data Analytics', 'Data and Knowledge Engineering']

def date_aleatoire(annee):
    return date(annee, 1, 1) + timedelta(days=random.randint(20, 330))

conferences = []        # (id, intitule, debut, fin, intro, id_ville, id_org, id_serie, id_editeur)
theme_principal = {}    # id_conf -> branche de thèmes principale
annees_conf = list(ANNEES)

# 1) La série SDH : une édition par an
for annee in ANNEES:
    cid = len(conferences) + 1
    d = date_aleatoire(annee)
    org = random.randint(1, len(organismes))
    conferences.append((cid, f'SDH {annee} – Spatial Data Handling', d, d + timedelta(days=2),
                        f'The {annee} edition of the International Symposium on Spatial Data Handling '
                        f'brings together researchers in GIScience and spatial databases.',
                        ville_organisme[org], org, 1, random.choice([1, 2])))
    theme_principal[cid] = ID_SPATIAL

# 2) Les autres conférences (une série n'a qu'une édition par an)
editions = {(1, annee) for annee in ANNEES}
intitules = {c[1] for c in conferences}
while len(conferences) < NB_CONFERENCES:
    cid = len(conferences) + 1
    # 15 premières conférences "libres" forcées dans l'année courante
    annee = ANNEE_COURANTE if cid <= len(ANNEES) + 15 else random.choice(annees_conf)
    d = date_aleatoire(annee)
    duree = random.choice([1, 2, 2, 3, 4])
    branche = random.choice(list(branches))
    id_serie = random.randint(2, len(series)) if random.random() < 0.7 else None
    if id_serie is not None and (id_serie, annee) in editions:
        id_serie = None                      # une seule édition par série et par an
    if id_serie is not None:
        editions.add((id_serie, annee))
        sigle = series[id_serie - 1][1]
        intitule = f'{sigle} {annee}'
        if sigle == 'TIME':
            branche = ID_TEMPORAL
    org = random.randint(1, len(organismes))
    ville = ville_organisme[org] if random.random() < 0.6 else random.randint(1, len(villes))
    if id_serie is None:
        # Conférence hors série : "Symposium on Data Science 2026" (le mot "Workshop"
        # est réservé aux vrais workshops). En cas de doublon, on précise la ville.
        sujet = random.choice(SUJETS.get(branche, SUJETS_GENERAUX))
        prefixe = random.choice(['International Conference on', 'Symposium on', 'Forum on', 'European Days on'])
        intitule = f'{prefixe} {sujet} {annee}'
        if intitule in intitules:
            intitule = f'{prefixe} {sujet} {annee} – {villes[ville - 1][1]}'
        if intitule in intitules:
            intitule = f'{prefixe} {sujet} {annee} – {villes[ville - 1][1]} ({cid})'
    intitules.add(intitule)
    editeur = random.randint(1, len(editeurs)) if random.random() < 0.85 else None
    conferences.append((cid, intitule, d, d + timedelta(days=duree), fake.paragraph(nb_sentences=4),
                        ville, org, id_serie, editeur))
    theme_principal[cid] = branche

inserer('conference', ['id_conf', 'intitule', 'date_debut', 'date_fin', 'texte_introductif',
                       'id_ville', 'id_organisme', 'id_serie', 'id_editeur'], conferences)

  conference             :   160 lignes


## 8. Workshops
Un workshop est d'abord une **conférence** (ligne dans `conference`), puis une ligne dans `workshop` qui le rattache à sa conférence mère.
Il a lieu **la veille ou le lendemain** de la conférence mère, dans la même ville (contrainte CI-2c : 7 jours maximum).

In [41]:
meres = random.sample([c[0] for c in conferences], NB_WORKSHOPS - 3)
meres += [1, 1, 2]                      # SDH a plusieurs workshops (démonstration requête 6)
conf_par_id = {c[0]: c for c in conferences}
ws_conf, ws_lien = [], []
for id_mere in meres:
    m = conf_par_id[id_mere]
    cid = len(conferences) + len(ws_conf) + 1
    if random.random() < 0.5:
        debut = m[2] - timedelta(days=1); fin = debut          # la veille
    else:
        debut = m[3] + timedelta(days=1); fin = debut + timedelta(days=random.choice([0, 1]))   # le lendemain
    branche = theme_principal[id_mere]
    sous_theme = random.choice([t for t in branches[branche] if t != branche] or [branche])
    lib = themes[sous_theme - 1][1]
    ws_conf.append((cid, f'Workshop on {lib} @ {m[1].split(" –")[0]}', debut, fin,
                    f'Satellite workshop of {m[1]} dedicated to {lib}.', m[5], m[6], None, m[8]))
    ws_lien.append((cid, id_mere))
    theme_principal[cid] = branche

inserer('conference', ['id_conf', 'intitule', 'date_debut', 'date_fin', 'texte_introductif',
                       'id_ville', 'id_organisme', 'id_serie', 'id_editeur'], ws_conf)
inserer('workshop', ['id_conf', 'id_conf_mere'], ws_lien)
toutes_conf = conferences + ws_conf

  conference             :    45 lignes
  workshop               :    45 lignes


## 9. Sessions et thèmes des sessions
- 2 à 5 sessions par conférence, 1 à 2 par workshop.
- Chaque session porte sur 1 à 3 thèmes, pris surtout dans la branche principale de la conférence.

In [42]:
sessions, session_themes = [], []
ids_workshops = {w[0] for w in ws_lien}
for c in toutes_conf:
    cid = c[0]
    nb = random.randint(1, 2) if cid in ids_workshops else random.randint(2, 5)
    branche = branches[theme_principal[cid]]
    for num in range(1, nb + 1):
        choix = random.sample(branche, k=min(len(branche), random.randint(1, 3)))
        if random.random() < 0.15:                        # un peu de transversalité
            choix.append(random.randint(1, len(themes)))
        choix = list(dict.fromkeys(choix))
        sessions.append((cid, num, f'Session {num} : {themes[choix[0] - 1][1]}'))
        for t in choix:
            session_themes.append((cid, num, t))

inserer('session', ['id_conf', 'num_session', 'intitule_session'], sessions)
inserer('session_theme', ['id_conf', 'num_session', 'id_theme'], session_themes)

  session                :   627 lignes
  session_theme          :  1335 lignes


## 10. Responsables, comités de programme et évaluateurs
- Chaque conférence a un **General chair** et un **Program Committee chair**, plus 0 à 3 autres responsables.
- Comité de programme de 8 à 15 personnes.
- Les évaluateurs d'une session sont **tirés parmi le comité de programme** de la conférence (sinon le trigger CI-1 refuserait l'insertion).
- Le professeur **Jean Dupont** (id 1) est responsable de plusieurs conférences SDH et membre de plusieurs comités (requête 9).

In [43]:
responsables, membres_pc, evaluations = set(), set(), set()
for c in toutes_conf:
    cid = c[0]
    candidats = random.sample(CHERCHEURS[2:], 25)    # Dupont et Alice sont gérés à part
    # Responsables
    responsables.add((candidats[0], cid, 1))                     # General chair
    responsables.add((candidats[1], cid, 2))                     # Program Committee chair
    for k, t in enumerate(random.sample(range(3, len(TYPES_RESP) + 1), random.randint(0, 3))):
        responsables.add((candidats[2 + k], cid, t))
    # Comité de programme
    pc = candidats[5:5 + random.randint(8, 15)]
    # Cas de test : Dupont dans les SDH et dans quelques autres comités ; Alice dans quelques comités
    if c[7] == 1 and c[2].year % 2 == 0:
        responsables.add((1, cid, 1))
    if (c[7] == 1 or random.random() < 0.05):
        pc.append(1)
    if random.random() < 0.06:
        pc.append(2)
    for pid in pc:
        membres_pc.add((pid, cid))
    # Évaluateurs : 2 à 4 membres du PC par session
    for (s_cid, num, _) in [s for s in sessions if s[0] == cid]:
        for pid in random.sample(pc, k=random.randint(2, 4)):
            evaluations.add((pid, cid, num))

inserer('responsable_de', ['id_personne', 'id_conf', 'id_type'], sorted(responsables))
inserer('membre_pc', ['id_personne', 'id_conf'], sorted(membres_pc))
inserer('evalue', ['id_personne', 'id_conf', 'num_session'], sorted(evaluations))

  responsable_de         :   704 lignes
  membre_pc              :  2396 lignes
  evalue                 :  1856 lignes


## 11. Spécifications des soumissions (appels à communication)
Pour chaque conférence, 2 à 5 catégories de soumission avec des échéances qui respectent :
`soumission < notification < version finale < début de la conférence` (CHECK + trigger CI-3).
Pas de nombre de pages pour *Panel*, *Tutorial* et *Workshop proposal* (sujet).

In [44]:
specifications = []
for c in toutes_conf:
    cid, debut = c[0], c[2]
    cats = ['Regular paper'] + random.sample(CATEGORIES[1:], random.randint(1, 4))
    if cid in ids_workshops:
        cats = random.sample(['Regular paper', 'Short paper', 'Poster'], random.randint(1, 2))
    for cat in cats:
        soumission   = debut - timedelta(days=random.randint(100, 160))
        notification = soumission + timedelta(days=random.randint(35, 50))
        finale       = notification + timedelta(days=random.randint(15, 30))
        if cat in SANS_PAGES:
            pages, police, taille, logiciel = None, None, None, None
        else:
            pages = {'Regular paper': random.choice([12, 14, 15, 16]), 'Short paper': random.choice([4, 6]),
                     'Poster': 2, 'Demo': 4}.get(cat, random.choice([6, 8, 10]))
            police = random.choice(['Times New Roman', 'Computer Modern', 'Arial'])
            taille = random.choice([10, 11, 12])
            logiciel = random.choice(['LaTeX', 'LaTeX', 'Word'])
        specifications.append((cid, CATEGORIES.index(cat) + 1, pages, police, taille, logiciel,
                               soumission, notification, finale))

inserer('specification', ['id_conf', 'id_categorie', 'nb_pages_max', 'police', 'taille_police', 'logiciel',
                          'date_limite_soumission', 'date_notification', 'date_limite_version_finale'],
        specifications)

  specification          :   647 lignes


## 12. Profils des utilisateurs (thèmes d'intérêt)
Chaque utilisateur s'intéresse à 1 à 4 thèmes. Alice s'intéresse à « Spatial databases » (requête 4 : elle doit voir les conférences sur GIS, R-tree… grâce à la hiérarchie).

In [45]:
profils = {(2, ID_SPATIAL)}
for (pid, login, _, role) in utilisateurs:
    if role == 'admin' or pid == 2:
        continue
    for t in random.sample(range(1, len(themes) + 1), random.randint(1, 4)):
        profils.add((pid, t))
inserer('utilisateur_theme', ['id_personne', 'id_theme'], sorted(profils))

  utilisateur_theme      :   181 lignes


## 13. Recaler les compteurs et valider
Les identifiants ont été fournis explicitement : il faut recaler les séquences `SERIAL`, sinon le prochain `INSERT` de l'interface tenterait l'id 1 (déjà pris).

In [46]:
for table, col in [('pays', 'id_pays'), ('ville', 'id_ville'), ('organisme', 'id_organisme'),
                   ('serie', 'id_serie'), ('editeur', 'id_editeur'), ('type_responsabilite', 'id_type'),
                   ('categorie_soumission', 'id_categorie'), ('theme', 'id_theme'),
                   ('conference', 'id_conf'), ('personne', 'id_personne')]:
    cur.execute(f"SELECT setval(pg_get_serial_sequence('conferences.{table}', '{col}'), "
                f"(SELECT MAX({col}) FROM {table}))")
conn.commit()
print('✅ Alimentation terminée et validée (COMMIT)')

✅ Alimentation terminée et validée (COMMIT)


## 14. Vérifications

In [47]:
for table in ['pays', 'ville', 'organisme', 'serie', 'editeur', 'type_responsabilite',
              'categorie_soumission', 'theme', 'conference', 'workshop', 'personne', 'utilisateur',
              'session', 'session_theme', 'responsable_de', 'membre_pc', 'evalue',
              'specification', 'utilisateur_theme']:
    cur.execute(f'SELECT COUNT(*) FROM {table}')
    print(f'{table:<22} {cur.fetchone()[0]:>6}')

pays                       15
ville                      41
organisme                  27
serie                       9
editeur                     8
type_responsabilite         7
categorie_soumission        9
theme                      43
conference                205
workshop                   45
personne                  366
utilisateur                71
session                   627
session_theme            1335
responsable_de            704
membre_pc                2396
evalue                   1856
specification             647
utilisateur_theme         181


In [48]:
# Aperçu des cas de test prévus pour les requêtes
verifs = {
    'R1 - conférences de l\'année courante':
        """SELECT COUNT(*) FROM conference WHERE date_debut >= date_trunc('year', CURRENT_DATE)
             AND date_debut < date_trunc('year', CURRENT_DATE) + INTERVAL '1 year'""",
    'R3 - conférences sur les bases spatiales (hiérarchie)':
        """SELECT COUNT(DISTINCT id_conf) FROM v_conference_themes
             WHERE id_theme IN (SELECT id_theme FROM fn_themes_descendants('spatial'))""",
    'R4 - conférences qui intéressent alice':
        "SELECT COUNT(DISTINCT id_conf) FROM v_utilisateur_conferences WHERE login = 'alice'",
    'R5 - conférences de la série SDH':
        "SELECT COUNT(*) FROM v_conference_complete WHERE serie_sigle = 'SDH'",
    'R6 - workshops de la conférence 1':
        "SELECT COUNT(*) FROM v_workshops WHERE id_conf_mere = 1",
    'R8 - utilisateurs aussi responsables':
        "SELECT COUNT(*) FROM v_utilisateurs_responsables",
    'R9 - conférences du professeur Dupont':
        "SELECT COUNT(DISTINCT id_conf) FROM v_personnes_responsables WHERE nom = 'Dupont' AND prenom = 'Jean'",
}
for libelle, sql in verifs.items():
    cur.execute(sql)
    print(f'{libelle:<55} {cur.fetchone()[0]:>4}')

R1 - conférences de l'année courante                      48
R3 - conférences sur les bases spatiales (hiérarchie)     55
R4 - conférences qui intéressent alice                    55
R5 - conférences de la série SDH                           8
R6 - workshops de la conférence 1                          2
R8 - utilisateurs aussi responsables                      25
R9 - conférences du professeur Dupont                     14


In [49]:
cur.close()
conn.close()
print('Connexion fermée')

Connexion fermée
